# Brain Tumor MRI Analyzer
**Goal:** Two-stage pipeline — EfficientNet classifier + U-Net segmentation with Grad-CAM explainability.

**Papers referenced:**
- Ronneberger et al., U-Net (2015)
- Menze et al., BraTS Challenge (2018/2020)
- Selvaraju et al., Grad-CAM (2017)

In [ ]:
# Colab setup cell
import os
import sys
import subprocess
from pathlib import Path

def _install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

try:
    import torch
except ImportError:
    _install('torch')
    _install('torchvision')
    _install('torchaudio')
    import torch

for package in ['numpy', 'matplotlib', 'seaborn', 'scikit-learn', 'pillow', 'opencv-python', 'ipykernel']:
    try:
        __import__(package.replace('-', '_'))
    except ImportError:
        _install(package)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

os.chdir('/content')
print('Setup complete. Current working directory:', Path.cwd())

## Colab Setup
If you open this notebook in Google Colab, run the next cell first. It installs the required packages, mounts Google Drive if needed, and sets a safe working path for the dataset and outputs.

In [4]:
# Section 2 — Setup & Imports
import os
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Reproducibility
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

ModuleNotFoundError: No module named 'numpy'

## Section 3 — Dataset
The dataset is already available in the archive folder:

- `archive (1)/Training`
- `archive (1)/Testing`

Use the `Training` split for EDA and model training, and `Testing` for final evaluation.

In [ ]:
# EDA: class distribution and sample images
from pathlib import Path
from utils.eda import get_class_distribution, plot_class_distribution, show_samples_grid

archive_root = Path('archive (1)')
train_dir = archive_root / 'Training'
test_dir = archive_root / 'Testing'

if not train_dir.exists() or not test_dir.exists():
    print('Expected archive layout not found.')
    print('Looking for:', train_dir)
    print('and:', test_dir)
else:
    train_counts, train_samples = get_class_distribution(train_dir)
    test_counts, _ = get_class_distribution(test_dir)

    print('Training class counts:', train_counts)
    print('Testing class counts:', test_counts)

    plot_class_distribution(train_counts)
    show_samples_grid(train_samples, per_class=3)

    print('\nClass counts per split:')
    print('Training:', train_counts)
    print('Testing:', test_counts)

## Section 4 — Preprocessing
Resize to 224x224, ImageNet normalization, and augmentation for training. See `utils/transforms.py` and `utils/preprocessing.py`.

In [ ]:
# Section 4 — Preprocessing & augmentations
from utils.preprocessing import build_transforms, load_sample_image, visualize_transform, get_dataloader_kwargs
from utils.dataset import BrainTumorImageDataset
from torch.utils.data import DataLoader, random_split
from utils.transforms import get_train_transforms, get_val_transforms

archive_root = Path('archive (1)')
train_dir = archive_root / 'Training'
test_dir = archive_root / 'Testing'
transforms_map = build_transforms()

print('Train directory:', train_dir)
print('Test directory:', test_dir)
print('Train transform:', transforms_map['train'])
print('Validation transform:', transforms_map['val'])

# Preview a sample image after preprocessing
if train_dir.exists():
    sample_path, sample_class = load_sample_image(train_dir)
    print('Sample image:', sample_path)
    print('Sample class:', sample_class)
    visualize_transform(sample_path, transforms_map['train'], title=f'Train transform preview: {sample_class}')

# Dataset wiring for later phases
# train_dataset = BrainTumorImageDataset(str(train_dir), transform=get_train_transforms())
# test_dataset = BrainTumorImageDataset(str(test_dir), transform=get_val_transforms())
# train_len = int(0.8 * len(train_dataset))
# val_len = len(train_dataset) - train_len
# train_ds, val_ds = random_split(train_dataset, [train_len, val_len], generator=torch.Generator().manual_seed(seed))
# loader_kwargs = get_dataloader_kwargs()
# train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
# val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
# test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

## Section 5 — EfficientNet Classifier
Train a pretrained EfficientNet-B0 on the 4 MRI classes, track loss/accuracy, and save the best checkpoint.

In [5]:
from utils.training import train_one_epoch, evaluate, save_checkpoint
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import DataLoader, random_split
from utils.dataset import BrainTumorImageDataset
from utils.transforms import get_train_transforms, get_val_transforms
from utils.preprocessing import get_dataloader_kwargs

archive_root = Path('archive (1)')
train_dir = archive_root / 'Training'
test_dir = archive_root / 'Testing'

# Dataset setup
train_dataset = BrainTumorImageDataset(str(train_dir), transform=get_train_transforms())
test_dataset = BrainTumorImageDataset(str(test_dir), transform=get_val_transforms())

train_len = int(0.8 * len(train_dataset))
val_len = len(train_dataset) - train_len
train_ds, val_ds = random_split(
    train_dataset,
    [train_len, val_len],
    generator=torch.Generator().manual_seed(seed),
)

loader_kwargs = get_dataloader_kwargs()
train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

# EfficientNet-B0 model
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 4)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
num_epochs = 10

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
}

best_val_acc = 0.0
best_checkpoint_path = Path('models') / 'efficientnet_classifier.pth'

for epoch in range(num_epochs):
    train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(train_metrics.loss)
    history['train_acc'].append(train_metrics.accuracy)
    history['val_loss'].append(val_metrics.loss)
    history['val_acc'].append(val_metrics.accuracy)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_metrics.loss:.4f} | Train Acc: {train_metrics.accuracy:.4f} | "
        f"Val Loss: {val_metrics.loss:.4f} | Val Acc: {val_metrics.accuracy:.4f}"
    )

    if val_metrics.accuracy > best_val_acc:
        best_val_acc = val_metrics.accuracy
        save_checkpoint(
            model,
            best_checkpoint_path,
            metadata={
                'epoch': epoch + 1,
                'val_accuracy': best_val_acc,
                'class_count': 4,
            },
        )
        print('Saved new best checkpoint to', best_checkpoint_path)

# Plot learning curves
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'], label='Validation')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train')
plt.plot(history['val_acc'], label='Validation')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()

KeyboardInterrupt: 

## Section 6 — Evaluation & Grad-CAM
Evaluate the classifier on the held-out test set, then visualize confusion matrix, classification report, per-class accuracy, and Grad-CAM overlays.

In [ ]:
from utils.evaluation import (
    collect_predictions,
    plot_confusion_matrix,
    get_classification_report_text,
    compute_per_class_accuracy,
    plot_per_class_accuracy,
)
from utils.gradcam import generate_gradcam_grid, find_last_conv_layer

class_names = ['Glioma', 'Meningioma', 'Pituitary', 'No Tumor']

# Evaluation on the test set
if 'test_loader' in globals():
    y_true, y_pred, y_prob = collect_predictions(model, test_loader, device)
    print(get_classification_report_text(y_true, y_pred, class_names))
    plot_confusion_matrix(y_true, y_pred, class_names, normalize=False)
    per_class_accuracy = compute_per_class_accuracy(y_true, y_pred, class_names)
    print('Per-class accuracy:', per_class_accuracy)
    plot_per_class_accuracy(per_class_accuracy)
else:
    print('test_loader not available yet. Run the classifier training cell first.')

# Grad-CAM examples from the test set
# This section assumes the test_loader is available and the model is trained.
if 'test_loader' in globals():
    model.eval()
    first_batch = next(iter(test_loader))
    sample_images, sample_labels = first_batch
    sample_images = sample_images[:4]
    sample_labels = sample_labels[:4]

    fig, axes = plt.subplots(len(sample_images), 3, figsize=(12, 4 * len(sample_images)))
    if len(sample_images) == 1:
        axes = np.expand_dims(axes, axis=0)

    for idx, image_tensor in enumerate(sample_images):
        image = image_tensor.permute(1, 2, 0).cpu().numpy()
        image = image - image.min()
        image = image / (image.max() + 1e-8)

        overlay, heatmap = generate_gradcam_grid(model, image_tensor, device)

        axes[idx, 0].imshow(image)
        axes[idx, 0].set_title(f'Original | True: {class_names[int(sample_labels[idx])]}')
        axes[idx, 0].axis('off')

        axes[idx, 1].imshow(heatmap)
        axes[idx, 1].set_title('Grad-CAM Heatmap')
        axes[idx, 1].axis('off')

        axes[idx, 2].imshow(overlay)
        axes[idx, 2].set_title('Overlay')
        axes[idx, 2].axis('off')

    plt.tight_layout()

## Section 7 — Grad-CAM
Use `pytorch-grad-cam`. See `utils/gradcam.py` for helper to generate overlays.

## Section 8 — U-Net Segmentation
Train a U-Net with a ResNet34 backbone on paired MRI images and binary tumor masks. This phase expects image/mask pairs, not only class folders.

In [ ]:
from utils.segmentation import TumorSegmentationDataset, evaluate_segmentation, plot_segmentation_triplet
import segmentation_models_pytorch as smp
import torch.nn as nn

# Update these paths once paired masks are available.
seg_images_root = archive_root / 'Training'
seg_masks_root = Path('archive (1)') / 'Masks'

seg_history = {
    'train_loss': [],
    'train_iou': [],
    'val_loss': [],
    'val_iou': [],
}

if seg_images_root.exists() and seg_masks_root.exists():
    seg_dataset = TumorSegmentationDataset(
        images_root=seg_images_root,
        masks_root=seg_masks_root,
        transform=get_val_transforms(),
    )
    print('Segmentation sample count:', len(seg_dataset))

    if len(seg_dataset) == 0:
        print('No paired image/mask files found. Check the mask folder structure.')
    else:
        seg_train_len = int(0.8 * len(seg_dataset))
        seg_val_len = len(seg_dataset) - seg_train_len
        seg_train_ds, seg_val_ds = random_split(
            seg_dataset,
            [seg_train_len, seg_val_len],
            generator=torch.Generator().manual_seed(seed),
        )

        seg_loader_kwargs = get_dataloader_kwargs()
        seg_train_loader = DataLoader(seg_train_ds, shuffle=True, **seg_loader_kwargs)
        seg_val_loader = DataLoader(seg_val_ds, shuffle=False, **seg_loader_kwargs)

        seg_model = smp.Unet(
            encoder_name='resnet34',
            encoder_weights='imagenet',
            in_channels=3,
            classes=1,
        ).to(device)
        seg_criterion = nn.BCEWithLogitsLoss()
        seg_optimizer = torch.optim.Adam(seg_model.parameters(), lr=1e-4)

        def train_one_epoch_segmentation(model, loader, criterion, optimizer, device):
            model.train()
            running_loss = 0.0
            iou_scores = []
            total = 0
            for images, masks in loader:
                images = images.to(device)
                masks = masks.to(device)

                optimizer.zero_grad(set_to_none=True)
                outputs = model(images)
                loss = criterion(outputs, masks)
                loss.backward()
                optimizer.step()

                probabilities = torch.sigmoid(outputs)
                predictions = (probabilities > 0.5).float()
                batch_iou = ((predictions * masks).sum(dim=(1, 2, 3)) + 1e-6) / (
                    (predictions + masks - predictions * masks).sum(dim=(1, 2, 3)) + 1e-6
                )
                iou_scores.append(batch_iou.mean().item())
                running_loss += loss.item() * images.size(0)
                total += images.size(0)

            return running_loss / max(total, 1), float(np.mean(iou_scores)) if iou_scores else 0.0

        num_seg_epochs = 5
        best_seg_iou = 0.0
        best_seg_checkpoint = Path('models') / 'unet_segmentation.pth'

        for epoch in range(num_seg_epochs):
            train_loss, train_iou = train_one_epoch_segmentation(
                seg_model, seg_train_loader, seg_criterion, seg_optimizer, device
            )
            val_iou = evaluate_segmentation(seg_model, seg_val_loader, device)

            seg_history['train_loss'].append(train_loss)
            seg_history['train_iou'].append(train_iou)
            seg_history['val_loss'].append(np.nan)
            seg_history['val_iou'].append(val_iou)

            print(
                f"Epoch {epoch + 1}/{num_seg_epochs} | "
                f"Train Loss: {train_loss:.4f} | Train IoU: {train_iou:.4f} | Val IoU: {val_iou:.4f}"
            )

            if val_iou > best_seg_iou:
                best_seg_iou = val_iou
                save_checkpoint(
                    seg_model,
                    best_seg_checkpoint,
                    metadata={'epoch': epoch + 1, 'val_iou': best_seg_iou},
                )
                print('Saved new best segmentation checkpoint to', best_seg_checkpoint)

        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.plot(seg_history['train_loss'], label='Train Loss')
        plt.title('Segmentation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(seg_history['train_iou'], label='Train IoU')
        plt.plot(seg_history['val_iou'], label='Val IoU')
        plt.title('Segmentation IoU')
        plt.xlabel('Epoch')
        plt.ylabel('IoU')
        plt.legend()
        plt.tight_layout()

        # Visualize one sample if available
        sample_image, sample_mask = seg_dataset[0]
        with torch.no_grad():
            sample_pred = torch.sigmoid(seg_model(sample_image.unsqueeze(0).to(device)))[0, 0].cpu().numpy()
        plot_segmentation_triplet(
            sample_image.permute(1, 2, 0).cpu().numpy(),
            sample_mask.squeeze(0).cpu().numpy(),
            sample_pred,
            title_prefix='Segmentation Demo',
        )
else:
    print('Segmentation folders not found yet.')
    print('Expected images at:', seg_images_root)
    print('Expected masks at:', seg_masks_root)
    print('Add paired mask files to continue with U-Net training.')

## Section 9 — Full Pipeline Demo
Run classifier → Grad-CAM → segmentation on selected test images and visualize side-by-side.

In [ ]:
from utils.pipeline import run_full_pipeline, load_classifier, load_segmentation_model, CLASS_NAMES
from utils.segmentation import plot_segmentation_triplet

classifier_checkpoint = Path('models') / 'efficientnet_classifier.pth'
segmentation_checkpoint = Path('models') / 'unet_segmentation.pth'

# Load saved models when checkpoints exist.
classifier_model = load_classifier(classifier_checkpoint, device)
segmentation_model = load_segmentation_model(segmentation_checkpoint, device)

pipeline_image_dir = test_dir if test_dir.exists() else train_dir
sample_image_paths = []
if pipeline_image_dir.exists():
    for class_dir in sorted([d for d in pipeline_image_dir.iterdir() if d.is_dir()]):
        class_images = sorted([
            p for p in class_dir.iterdir()
            if p.suffix.lower() in ('.jpg', '.jpeg', '.png')
        ])
        if class_images:
            sample_image_paths.append(class_images[0])
        if len(sample_image_paths) >= 4:
            break

if not sample_image_paths:
    print('No sample images found for the pipeline demo.')
else:
    fig, axes = plt.subplots(len(sample_image_paths), 3, figsize=(14, 4 * len(sample_image_paths)))
    if len(sample_image_paths) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_index, image_path in enumerate(sample_image_paths):
        result = run_full_pipeline(
            image_path=image_path,
            classifier=classifier_model,
            device=device,
            segmentation_model=segmentation_model,
        )

        original_image = np.array(result['original_image'])
        axes[row_index, 0].imshow(original_image)
        axes[row_index, 0].set_title(
            f"Original | True folder: {image_path.parent.name}"
        )
        axes[row_index, 0].axis('off')

        axes[row_index, 1].imshow(result['gradcam_heatmap'])
        axes[row_index, 1].set_title(
            f"Grad-CAM | {result['prediction_label']} ({result['confidence']:.2f})"
        )
        axes[row_index, 1].axis('off')

        if result['segmentation_mask'] is not None:
            axes[row_index, 2].imshow(result['segmentation_mask'], cmap='gray')
            axes[row_index, 2].set_title('Segmentation Mask')
        else:
            axes[row_index, 2].text(
                0.5,
                0.5,
                'Segmentation checkpoint not found',
                ha='center',
                va='center',
                fontsize=11,
            )
            axes[row_index, 2].set_title('Segmentation')
        axes[row_index, 2].axis('off')

    plt.tight_layout()

    print('Pipeline demo complete.')
    print('Classifier checkpoint:', classifier_checkpoint)
    print('Segmentation checkpoint:', segmentation_checkpoint)
    print('Segmentation model loaded:', segmentation_model is not None)

In [ ]:
# Pipeline readiness check before moving to the UI
required_artifacts = {
    'classifier_checkpoint': Path('models') / 'efficientnet_classifier.pth',
    'segmentation_checkpoint': Path('models') / 'unet_segmentation.pth',
}

for name, artifact_path in required_artifacts.items():
    print(f'{name}:', 'FOUND' if artifact_path.exists() else f'MISSING -> {artifact_path}')

print('Training cells must be executed before the UI can use real predictions.')

## Section 10 — Conclusion & Next Steps
The notebook now contains the full research pipeline: data loading, preprocessing, classifier training, evaluation with Grad-CAM, segmentation, and an end-to-end demo. The remaining work before the UI is to verify that the trained checkpoints exist and that the pipeline runs on sample images after executing the training cells.

## Section 10 — Conclusion
Summarize results, limitations, and future extensions (BraTS 3D, multi-modal MRI).